# 5 — Split-cell-line reactome terms (quick check)

Per-cell-line **split** version of the enrichment analysis from notebook 3.
Instead of pooling the "small / close-enhancer + up-regulated" gene sets from
both cell lines of a comparison, here each **ordered** cell-line pair is run on
its own so we can see what each side produces in isolation.

We run **12 experiments** = 6 ordered cell-line pairs × 2 jump definitions:

* **with mids** — `small → mid` **and** `small → large` (i.e. *cell1 small vs cell2 mid/large*)
* **without mids** — `small → large` only (i.e. *cell1 small vs cell2 large*)

For every experiment we take only the **cell1** gene set (up-regulated in cell1
*and* close/small in cell1, having jumped to mid/large in cell2), enrich against
**Reactome_Pathways_2024**, and print the terms as **text** (no figure yet).

In [1]:
import os.path

import pyranges

gencode = pyranges.read_gtf("../../data/gencode.v40.basic.annotation.gtf")

genes_from_gencode = (
    gencode
    [(gencode['Feature'] == 'gene')]
    [['Chromosome', 'Start', 'End', 'gene_id']]
)

genes_from_gencode['gene_id'] = genes_from_gencode['gene_id'].str.split('.').str[0]

genes_from_gencode

,Chromosome,Start,End,gene_id
0,chr1,11868,14409,ENSG00000223972
12,chr1,14403,29570,ENSG00000227232
25,chr1,17368,17436,ENSG00000278267
28,chr1,29553,31109,ENSG00000243485
36,chr1,30365,30503,ENSG00000284332
...,...,...,...,...
1926897,chrM,14148,14673,ENSG00000198695
1926902,chrM,14673,14742,ENSG00000210194
1926905,chrM,14746,15887,ENSG00000198727
1926910,chrM,15887,15953,ENSG00000210195


In [2]:
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

with open("../../data/rnaseq/gm12878_read_count.txt", "r") as f:
    f.readline()
    gm12878_counts_df = pd.read_csv(f, index_col=0, sep="\t")


with open("../../data/rnaseq/h1esc_read_count.txt", "r") as f:
    f.readline()
    h1esc_counts_df = pd.read_csv(f, index_col=0, sep="\t")

with open("../../data/rnaseq/hffc6_read_count.txt", "r") as f:
    f.readline()
    hffc6_counts_df = pd.read_csv(f, index_col=0, sep="\t")

gm12878_counts_df = gm12878_counts_df[['ENCFF800HIP.filtered.bam', 'ENCFF991KKX.filtered.bam']].T
h1esc_counts_df = h1esc_counts_df[['ENCFF675NTU.filtered.bam', 'ENCFF379NOY.filtered.bam']].T
hffc6_counts_df = hffc6_counts_df[['ENCFF307KUI.filtered.bam', 'ENCFF294BSI.filtered.bam', 'ENCFF937TEI.filtered.bam']].T

def compute_deseq_stats_for_2_celllines(cellline1_counts_df, cellline2_counts_df, cellline1_name="CellLine1", cellline2_name="CellLine2"):
    cl1_vs_cl2_counts_df = pd.concat(
        [cellline1_counts_df, cellline2_counts_df]
    )

    cl1_vs_cl2_metadata_df = pd.DataFrame(
        {
            "condition": [cellline1_name] * cellline1_counts_df.shape[0] + [cellline2_name] * cellline2_counts_df.shape[0]
        },
        index=cl1_vs_cl2_counts_df.index,
    )

    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet(
        counts=cl1_vs_cl2_counts_df,
        metadata=cl1_vs_cl2_metadata_df,
        design="~condition",
        refit_cooks=True,
        inference=inference,
    )

    dds.deseq2()
    stats = DeseqStats(dds, contrast=["condition", cellline2_name, cellline1_name])
    stats.summary()

    stats.lfc_shrink(coeff=f"condition[T.{cellline2_name}]")

    results_df = stats.results_df.dropna()
    results_df['symbol'] = results_df.index
    return results_df

if os.path.exists("../../data/deseq/gm12878_vs_h1esc_results.parquet"):
    gm12878_vs_h1esc_results_df = pd.read_parquet("../../data/deseq/gm12878_vs_h1esc_results.parquet")
else:
    gm12878_vs_h1esc_results_df = compute_deseq_stats_for_2_celllines(
        gm12878_counts_df,
        h1esc_counts_df,
        cellline1_name="GM12878",
        cellline2_name="H1ESC"
    )
    gm12878_vs_h1esc_results_df.to_parquet("../../data/deseq/gm12878_vs_h1esc_results.parquet")

if os.path.exists("../../data/deseq/h1esc_vs_hffc6_results.parquet"):
    h1esc_vs_hffc6_results_df = pd.read_parquet("../../data/deseq/h1esc_vs_hffc6_results.parquet")
else:
    h1esc_vs_hffc6_results_df = compute_deseq_stats_for_2_celllines(
        h1esc_counts_df,
        hffc6_counts_df,
        cellline1_name="H1ESC",
        cellline2_name="HFFC6"
    )
    h1esc_vs_hffc6_results_df.to_parquet("../../data/deseq/h1esc_vs_hffc6_results.parquet")

if os.path.exists("../../data/deseq/gm12878_vs_hffc6_results.parquet"):
    gm12878_vs_hffc6_results_df = pd.read_parquet("../../data/deseq/gm12878_vs_hffc6_results.parquet")
else:
    gm12878_vs_hffc6_results_df = compute_deseq_stats_for_2_celllines(
        gm12878_counts_df,
        hffc6_counts_df,
        cellline1_name="GM12878",
        cellline2_name="HFFC6",
    )
    gm12878_vs_hffc6_results_df.to_parquet("../../data/deseq/gm12878_vs_hffc6_results.parquet")

In [3]:
# get_comparison_table_2_cell_lines expects a DESeq frame oriented so that
# log2FoldChange > 0  <=>  up-regulated in cell2 (the second name passed).
# We have GM_vs_H1, GM_vs_HFFC6 and H1_vs_HFFC6; build the three reverse
# orientations by flipping the sign of the log2 fold-change.
def reverse_contrast(df):
    reversed_df = df.copy()
    reversed_df['log2FoldChange'] = -reversed_df['log2FoldChange']
    return reversed_df

h1esc_vs_gm12878_results_df = reverse_contrast(gm12878_vs_h1esc_results_df)
hffc6_vs_gm12878_results_df = reverse_contrast(gm12878_vs_hffc6_results_df)
hffc6_vs_h1esc_results_df   = reverse_contrast(h1esc_vs_hffc6_results_df)

In [4]:
gene_to_closest_avg_enhancer_all = pd.read_parquet("../../data/whole_chromosomes/genes_all_cell_lines_v3.snappy.parquet")
gm12878_gene_to_closest_avg_enhancer_all = gene_to_closest_avg_enhancer_all[gene_to_closest_avg_enhancer_all['cell_line'] == 'GM12878'].set_index('gene_id').drop(columns=['cell_line'])
h1esc_gene_to_closest_avg_enhancer_all = gene_to_closest_avg_enhancer_all[gene_to_closest_avg_enhancer_all['cell_line'] == 'H1ESC'].set_index('gene_id').drop(columns=['cell_line'])
hffc6_gene_to_closest_avg_enhancer_all = gene_to_closest_avg_enhancer_all[gene_to_closest_avg_enhancer_all['cell_line'] == 'HFFC6'].set_index('gene_id').drop(columns=['cell_line'])

def add_chromosome_and_quartiles(df, name):
    df = df.merge(genes_from_gencode[['gene_id', 'Chromosome']], left_index=True, right_on='gene_id', how='left').set_index('gene_id')

    def compute_quartiles(group):
        q1 = group['min_dist'].quantile(0.33)
        q3 = group['min_dist'].quantile(0.67)
        def label_quartile(x):
            if x <= q1:
                return 'small'
            elif x <= q3:
                return 'mid'
            else:
                return 'large'
        group['quartile_cat'] = group['min_dist'].apply(label_quartile)
        return group

    df = df.groupby('Chromosome', group_keys=False).apply(compute_quartiles, include_groups=False)
    return df

gm12878_gene_to_closest_avg_enhancer_all = add_chromosome_and_quartiles(gm12878_gene_to_closest_avg_enhancer_all, 'gm12878')
hffc6_gene_to_closest_avg_enhancer_all = add_chromosome_and_quartiles(hffc6_gene_to_closest_avg_enhancer_all, 'hffc6')
h1esc_gene_to_closest_avg_enhancer_all = add_chromosome_and_quartiles(h1esc_gene_to_closest_avg_enhancer_all, 'h1esc')

In [5]:
import numpy as np
from gprofiler import GProfiler

gp = GProfiler(
    user_agent='enhancer3d',
    return_dataframe=True,
)

def get_comparison_table_2_cell_lines(
    cell1_distances_df,
    cell2_distances_df,
    deseq_results_df,
    cell1_name,
    cell2_name,
    log2fc_threshold=2,
    padj_threshold=0.05,
    enable_jump_filtering=True,
    enable_expression_filtering=True,
    cell1_jump_categories=(
        "Increased (small to large)",
        "Increased (small to mid)",
    ),
    cell2_jump_categories=(
        "Decreased (large to small)",
        "Decreased (mid to small)",
    )
):
    cell1_jump_categories = list(cell1_jump_categories)
    cell2_jump_categories = list(cell2_jump_categories)

    deseq_results_df = deseq_results_df.copy()
    deseq_results_df.index = deseq_results_df.index.str.split('.').str[0]

    deseq_results_df = deseq_results_df[
        (deseq_results_df["padj"] < padj_threshold)
        & (deseq_results_df["log2FoldChange"].abs() > log2fc_threshold)
    ]

    cell1_distances_df = cell1_distances_df.copy()
    cell1_distances_df['norm_min_dist'] = (
        (cell1_distances_df['min_dist'] - cell1_distances_df['min_dist'].min()) / (cell1_distances_df['min_dist'].max() - cell1_distances_df['min_dist'].min())
    )

    cell2_distances_df = cell2_distances_df.copy()
    cell2_distances_df['norm_min_dist'] = (
        (cell2_distances_df['min_dist'] - cell2_distances_df['min_dist'].min()) / (cell2_distances_df['min_dist'].max() - cell2_distances_df['min_dist'].min())
    )

    combined_df = cell1_distances_df.merge(
        cell2_distances_df,
        left_index=True,
        right_index=True,
        how="inner",
        suffixes=(f"_{cell1_name}", f"_{cell2_name}")
    )

    combined_df["log_dist_ratio"] = np.log2(combined_df[f"min_dist_{cell1_name}"] / combined_df[f"min_dist_{cell2_name}"])
    combined_df["dist_diff"] = combined_df[f"min_dist_{cell1_name}"] - combined_df[f"min_dist_{cell2_name}"]

    combined_df["norm_dist_diff"] = combined_df[f"norm_min_dist_{cell1_name}"] - combined_df[f"norm_min_dist_{cell2_name}"]
    combined_df["log_norm_dist_ratio"] = np.log2(combined_df[f"norm_min_dist_{cell1_name}"] / combined_df[f"norm_min_dist_{cell2_name}"])

    def get_quartile_change(row):
        q1 = row[f"quartile_cat_{cell1_name}"]
        q2 = row[f"quartile_cat_{cell2_name}"]

        if q1 == q2:
            return "No change"

        categories = ["small", "mid", "large"]
        idx1 = categories.index(q1)
        idx2 = categories.index(q2)

        if idx1 > idx2:
            return f"Decreased ({q1} to {q2})"
        else:
            return f"Increased ({q1} to {q2})"

    combined_df["quartile_change"] = combined_df.apply(get_quartile_change, axis=1)

    merged_df = combined_df.merge(
        deseq_results_df[["log2FoldChange", "padj"]],
        left_index=True,
        right_index=True,
        how="inner"
    )

    merged_df_for_genes = merged_df.copy()

    if enable_jump_filtering and enable_expression_filtering:
        cell_line_1_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["log2FoldChange"] < -log2fc_threshold)
            & (merged_df_for_genes["quartile_change"].isin(cell1_jump_categories))
        ]
    elif enable_jump_filtering:
        cell_line_1_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["quartile_change"].isin(cell1_jump_categories))
        ]
    elif enable_expression_filtering:
        cell_line_1_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["log2FoldChange"] < -log2fc_threshold)
        ]

    if enable_jump_filtering and enable_expression_filtering:
        cell_line_2_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["log2FoldChange"] > log2fc_threshold)
            & (merged_df_for_genes["quartile_change"].isin(cell2_jump_categories))
        ]
    elif enable_jump_filtering:
        cell_line_2_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["quartile_change"].isin(cell2_jump_categories))
        ]
    elif enable_expression_filtering:
         cell_line_2_upregulated_jumper_genes = merged_df_for_genes[
            (merged_df_for_genes["log2FoldChange"] > log2fc_threshold)
        ]

    return cell_line_1_upregulated_jumper_genes, cell_line_2_upregulated_jumper_genes

In [6]:
import gseapy

def get_enrichment_results(gene_list, gene_sets):
    columns = ['Term', 'P-value', 'Adjusted P-value', 'Overlap', 'Combined Score', 'Genes']

    enr = gseapy.enrichr(
        gene_list=gene_list,
        gene_sets=gene_sets,
        organism='Human',
        outdir=None,
        cutoff=0.05
    )

    # Enrichr can return an empty frame (no terms / API hiccup); keep the schema stable.
    if enr.results is None or enr.results.empty:
        return pd.DataFrame(columns=columns + ['Genes Ratio', 'Genes Number'])

    results = enr.results[columns].copy()
    results['Genes Ratio'] = results['Genes'].apply(lambda x: len(x.split(';')) / len(gene_list))
    results['Genes Number'] = results['Genes'].apply(lambda x: len(x.split(';')))
    return results.reset_index(drop=True)

In [7]:
def get_cell1_small_gene_names(cell1_distances_df, cell2_distances_df, deseq_results_df,
                               cell1_name, cell2_name, include_mid=True):
    """Genes up-regulated in cell1 that are SMALL (close enhancer) in cell1 and
    jumped to mid/large in cell2. With include_mid=False, only small->large counts."""
    cell1_jump_categories = ["Increased (small to large)"]
    if include_mid:
        cell1_jump_categories.append("Increased (small to mid)")

    cell1_genes_df, _ = get_comparison_table_2_cell_lines(
        cell1_distances_df,
        cell2_distances_df,
        deseq_results_df,
        cell1_name,
        cell2_name,
        cell1_jump_categories=tuple(cell1_jump_categories),
    )

    if cell1_genes_df.empty:
        return []

    names = (
        gp.convert(organism='hsapiens', query=cell1_genes_df.index.tolist())
        [['converted', 'name']]
        .set_index('converted', drop=True)
    )
    return list(filter(None, names['name'].tolist()))


distance_dfs = {
    "GM12878": gm12878_gene_to_closest_avg_enhancer_all,
    "H1ESC":   h1esc_gene_to_closest_avg_enhancer_all,
    "HFFC6":   hffc6_gene_to_closest_avg_enhancer_all,
}

# 6 ordered (cell1, cell2) pairs -> the contrast oriented so log2FC>0 == up in cell2.
experiments = [
    ("GM12878", "H1ESC",   gm12878_vs_h1esc_results_df),
    ("GM12878", "HFFC6",   gm12878_vs_hffc6_results_df),
    ("H1ESC",   "GM12878", h1esc_vs_gm12878_results_df),
    ("H1ESC",   "HFFC6",   h1esc_vs_hffc6_results_df),
    ("HFFC6",   "GM12878", hffc6_vs_gm12878_results_df),
    ("HFFC6",   "H1ESC",   hffc6_vs_h1esc_results_df),
]


def run_split_experiments(include_mid):
    target = "mid/large" if include_mid else "large"
    results = {}
    for cell1, cell2, deseq_df in experiments:
        label = f"{cell1} small vs {cell2} {target}"
        gene_names = get_cell1_small_gene_names(
            distance_dfs[cell1], distance_dfs[cell2], deseq_df,
            cell1, cell2, include_mid=include_mid,
        )
        reactomes = get_enrichment_results(gene_names, gene_sets='Reactome_Pathways_2024')
        results[label] = (gene_names, reactomes)

        print("=" * 90)
        print(f"{label}   ({len(gene_names)} genes, {len(reactomes)} reactome terms)")
        print("=" * 90)
        for term in reactomes['Term']:
            print(f"  {term}")
        print()
    return results

## Set A — *with mids* (cell1 small vs cell2 **mid/large**)

`small → mid` and `small → large` jumps.

In [8]:
results_with_mid = run_split_experiments(include_mid=True)

GM12878 small vs H1ESC mid/large   (242 genes, 602 reactome terms)
  Cytokine Signaling in Immune System
  Immune System
  Signaling by Interleukins
  Interleukin-3, Interleukin-5 and GM-CSF Signaling
  GPVI-mediated Activation Cascade
  Innate Immune System
  Regulation of Signaling by CBL
  Interleukin-2 Family Signaling
  Toll-like Receptor Cascades
  Toll Like Receptor 4 (TLR4) Cascade
  Signaling by CSF1 (M-CSF) in Myeloid Cells
  Signaling by the B Cell Receptor (BCR)
  Antigen Activates B Cell Receptor (BCR) Leading to Generation of Second Messengers
  Interferon Gamma Signaling
  Adaptive Immune System
  TAK1-dependent IKK and NF-kappa-B Activation
  Interleukin-4 and Interleukin-13 Signaling
  RUNX3 Regulates Immune Response and Cell Migration
  FLT3 Signaling Through SRC Family Kinases
  Toxicity of Botulinum Toxin Type D (botD)
  Toxicity of Botulinum Toxin Type F (botF)
  Regulation of IFNA IFNB Signaling
  Signaling by Erythropoietin
  Interleukin-7 Signaling
  Platelet Ac

/opt/anaconda3/envs/enhancer3D/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


H1ESC small vs GM12878 mid/large   (297 genes, 748 reactome terms)
  Neuronal System
  Transmission Across Chemical Synapses
  Neurotransmitter Receptors and Postsynaptic Signal Transmission
  Epithelial-Mesenchymal Transition (EMT) During Gastrulation
  Laminin Interactions
  Repression of WNT Target Genes
  Extracellular Matrix Organization
  SUMOylation of DNA Methylation Proteins
  EPH-Ephrin Signaling
  Activation of NMDA Receptors and Postsynaptic Events
  Ras Activation Upon Ca2+ Influx Through NMDA Receptor
  VLDL Clearance
  High Laminar Flow Shear Stress Activates Signaling by PIEZO1 and PECAM1 CDH5 KDR in Endothelial Cell
  RHO GTPases Activate PAKs
  Activation of GABAB Receptors
  GABA B Receptor Activation
  SHC1 Events in ERBB2 Signaling
  G Alpha (Z) Signalling Events
  Post NMDA Receptor Activation Events
  EGFR Transactivation by Gastrin
  Cell Junction Organization
  CREB1 Phosphorylation Through NMDA Receptor-Mediated Activation of RAS Signaling
  Signaling by Rho G

/opt/anaconda3/envs/enhancer3D/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


H1ESC small vs HFFC6 mid/large   (310 genes, 633 reactome terms)
  IRS Activation
  Semaphorin Interactions
  Other Semaphorin Interactions
  AKT Phosphorylates Targets in the Nucleus
  Adenylate Cyclase Activating Pathway
  Regulation of FOXO Transcriptional Activity by Acetylation
  Signal Attenuation
  Regulation of Localization of FOXO Transcription Factors
  Reversible Hydration of Carbon Dioxide
  RHO GTPase Cycle
  Nephron Development
  Adenylate Cyclase Inhibitory Pathway
  RAC1 GTPase Cycle
  High Laminar Flow Shear Stress Activates Signaling by PIEZO1 and PECAM1 CDH5 KDR in Endothelial Cell
  VLDLR Internalisation and Degradation
  Vasopressin Regulates Renal Water Homeostasis via Aquaporins
  Regulation of Endogenous Retroelements by KRAB-ZFP Proteins
  CDC42 GTPase Cycle
  PKA Activation in Glucagon Signalling
  Signaling by MET
  PKA Activation
  Kidney Development
  Signaling by Rho GTPases
  PKA-mediated Phosphorylation of CREB
  Sema4D Induced Cell Migration and Growth-

## Set B — *without mids* (cell1 small vs cell2 **large**)

`small → large` jumps only.

In [9]:
results_without_mid = run_split_experiments(include_mid=False)

GM12878 small vs H1ESC large   (60 genes, 264 reactome terms)
  Metabolism of Lipids
  Cytokine Signaling in Immune System
  G Beta Gamma Signalling Through PI3Kgamma
  Membrane Trafficking
  Intra-Golgi and Retrograde Golgi-to-ER Traffic
  Phospholipid Metabolism
  Signaling by CSF1 (M-CSF) in Myeloid Cells
  G-protein Beta Gamma Signalling
  GPVI-mediated Activation Cascade
  FLT3 Signaling
  Rab Regulation of Trafficking
  Vesicle-mediated Transport
  Platelet Activation, Signaling and Aggregation
  Intra-Golgi Traffic
  Signaling by Interleukins
  Synthesis of PIPs at the Plasma Membrane
  Biosynthesis of E-series 18(S)-resolvins
  Toxicity of Botulinum Toxin Type E (botE)
  FLT3 Signaling Through SRC Family Kinases
  GDP-fucose Biosynthesis
  Biosynthesis of EPA-derived SPMs
  Biosynthesis of Lipoxins (LX)
  Toxicity of Botulinum Toxin Type D (botD)
  Toxicity of Botulinum Toxin Type F (botF)
  RUNX3 Regulates Immune Response and Cell Migration
  Immune System
  Biosynthesis of Ma

/opt/anaconda3/envs/enhancer3D/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


H1ESC small vs GM12878 large   (91 genes, 332 reactome terms)
  Laminin Interactions
  Tight Junction Interactions
  Extracellular Matrix Organization
  Cell-cell Junction Organization
  Cell Junction Organization
  SUMOylation of DNA Methylation Proteins
  Non-integrin membrane-ECM Interactions
  RHO GTPases Activate ROCKs
  RHO GTPases Activate CIT
  Sema4D Induced Cell Migration and Growth-Cone Collapse
  RHO GTPases Activate PAKs
  SHC1 Events in ERBB2 Signaling
  Sema4D in Semaphorin Signaling
  Cell-Cell Communication
  Syndecan Interactions
  EPHA-mediated Growth Cone Collapse
  EPH-Ephrin Signaling
  Activation of Matrix Metalloproteinases
  RET Signaling
  Gap Junction Trafficking
  Signal Transduction
  Signaling by ERBB2
  Gap Junction Trafficking and Regulation
  Electric Transmission Across Gap Junctions
  Transmission Across Electrical Synapses
  Disinhibition of SNARE Formation
  VLDL Assembly
  Degradation of the Extracellular Matrix
  Signaling by Non-Receptor Tyrosine

/opt/anaconda3/envs/enhancer3D/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


H1ESC small vs HFFC6 large   (95 genes, 282 reactome terms)
  Cell-cell Junction Organization
  Cell Junction Organization
  L1CAM Interactions
  Serotonin and Melatonin Biosynthesis
  Sodium-coupled Sulphate, Di- and Tri-Carboxylate Transporters
  IRS Activation
  Defective LFNG Causes SCDO3
  Neurexins and Neuroligins
  Diseases of Glycosylation
  Pre-NOTCH Processing in the Endoplasmic Reticulum
  Signaling by ERBB4
  Epithelial-Mesenchymal Transition (EMT) During Gastrulation
  CREB Phosphorylation
  NCAM Signaling for Neurite Out-Growth
  Cell-Cell Communication
  PIP3 Activates AKT Signaling
  Noncanonical Activation of NOTCH3
  Insulin-like Growth Factor-2 mRNA Binding Proteins (IGF2BPs IMPs VICKZs) Bind RNA
  Defective CHST6 Causes MCDC1
  Diseases Associated With O-glycosylation of Proteins
  Sperm Motility And Taxes
  RHO GTPases Activate Rhotekin and Rhophilins
  Signal Attenuation
  PI3K Events in ERBB4 Signaling
  InlA-mediated Entry of Listeria Monocytogenes Into Host Cel

## Export terms to `export/`

* `split_celllines_reactome_terms.txt` — human-readable term lists for **all**
  terms, both sets, all 12 experiments, with the **FDR** (Enrichr Adjusted
  P-value) shown next to each term.
* `split_celllines_reactome_terms_fdr_significant.txt` — same layout but keeping
  **only terms with FDR < 0.05**.
* `split_celllines_reactomes.csv` — tidy long-format table (Experiment, Jump set,
  Gene count + the full Enrichr columns, incl. Adjusted P-value/FDR) for
  downstream use.

In [10]:
# FDR in Enrichr output == the "Adjusted P-value" column (Benjamini-Hochberg).
FDR_COL = "Adjusted P-value"
FDR_THRESHOLD = 0.05


def _write_terms(fh, exp_label, gene_names, reactomes, fdr_threshold=None):
    df = reactomes
    if fdr_threshold is not None:
        df = df[df[FDR_COL] < fdr_threshold]
    fh.write("=" * 90 + "\n")
    fh.write(f"{exp_label}   ({len(gene_names)} genes, {len(df)} reactome terms)\n")
    fh.write("=" * 90 + "\n")
    for _, row in df.iterrows():
        fh.write(f"  {row['Term']}\t(FDR={row[FDR_COL]:.3g})\n")
    fh.write("\n")


def export_results(all_sets, txt_path, csv_path, txt_sig_path):
    # all_sets: list of (set_label, results_dict{exp_label: (gene_names, reactomes_df)})
    # txt_path     -> all terms, with FDR shown next to each term
    # txt_sig_path -> only terms with FDR < FDR_THRESHOLD, with FDR shown
    for path, threshold in ((txt_path, None), (txt_sig_path, FDR_THRESHOLD)):
        with open(path, "w") as fh:
            if threshold is not None:
                fh.write(f"# Only Reactome terms with FDR < {threshold}\n\n")
            for set_label, results in all_sets:
                fh.write("#" * 90 + "\n")
                fh.write(f"# {set_label}\n")
                fh.write("#" * 90 + "\n\n")
                for exp_label, (gene_names, reactomes) in results.items():
                    _write_terms(fh, exp_label, gene_names, reactomes, fdr_threshold=threshold)

    frames = []
    for set_label, results in all_sets:
        for exp_label, (gene_names, reactomes) in results.items():
            frame = reactomes.copy()
            frame.insert(0, "Experiment", exp_label)
            frame.insert(1, "Jump set", set_label)
            frame.insert(2, "Gene count", len(gene_names))
            frames.append(frame)
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    combined.to_csv(csv_path, index=False)
    return combined


split_reactomes_combined = export_results(
    [
        ("with mids (small -> mid/large)", results_with_mid),
        ("without mids (small -> large)", results_without_mid),
    ],
    "export/split_celllines_reactome_terms.txt",
    "export/split_celllines_reactomes.csv",
    "export/split_celllines_reactome_terms_fdr_significant.txt",
)
print("wrote export/split_celllines_reactome_terms.txt")
print("wrote export/split_celllines_reactome_terms_fdr_significant.txt")
print("wrote export/split_celllines_reactomes.csv")
split_reactomes_combined.head()

wrote export/split_celllines_reactome_terms.txt
wrote export/split_celllines_reactome_terms_fdr_significant.txt
wrote export/split_celllines_reactomes.csv


,Experiment,Jump set,Gene count,Term,P-value,Adjusted P-value,Overlap,Combined Score,Genes,Genes Ratio,Genes Number
0,GM12878 small vs H1ESC mid/large,with mids (small -> mid/large),242,Cytokine Signaling in Immune System,3.342591e-11,2.012240e-08,33/776,107.900618,CD40;CIITA;PTAFR;NLRC5;PIK3CD;USP18;ALOX5;GRAP...,0.136364,33
1,GM12878 small vs H1ESC mid/large,with mids (small -> mid/large),242,Immune System,8.416147e-11,2.533260e-08,58/2150,69.359818,CD40;CIITA;ARPC1B;ICAM3;CGAS;PIK3CD;ITGAL;ADGR...,0.239669,58
2,GM12878 small vs H1ESC mid/large,with mids (small -> mid/large),242,Signaling by Interleukins,7.313282e-09,1.467532e-06,22/452,92.721605,IL15RA;MEF2C;SYK;PTAFR;NLRC5;PIK3CD;IL16;USP18...,0.090909,22
3,GM12878 small vs H1ESC mid/large,with mids (small -> mid/large),242,"Interleukin-3, Interleukin-5 and GM-CSF Signaling",9.446440e-07,1.421689e-04,7/48,217.425398,HCK;TEC;SYK;BLNK;PIK3CD;PTPN6;JAK3,0.028926,7
4,GM12878 small vs H1ESC mid/large,with mids (small -> mid/large),242,GPVI-mediated Activation Cascade,2.168132e-06,2.610431e-04,6/35,246.704439,SYK;PLCG2;RHOG;PTPN6;PIK3R6;PIK3R5,0.024793,6
